# FinDrama LOB Pretraining (FI-2010, Mamba3 MIMO)

One-click pretraining of the Mamba3 MIMO world model on the FI-2010 Nasdaq Helsinki limit order book benchmark (Ntakaris et al. 2018). Just paste your `HF_TOKEN` into Colab Secrets and run all cells.

Works on H100, A100, L4, T4 - any CUDA GPU. The Mamba3 MIMO TileLang kernels run fastest on H100 (sm_90); other GPUs fall back to the slower Python reference path. Pick a high-VRAM Colab runtime for best throughput.

Switch `DATASET = "polymarket"` in the first cell to fall back to the original Polymarket pipeline.


In [ ]:
import os, sys, subprocess
from pathlib import Path

# Runtime detection. Colab: ephemeral /content; cold env needs everything installed.
# Local: persistent home dir; user manages their own conda env.
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

REPO_URL = "https://github.com/Ruuudy1/FinDrama.git"
BRANCH = "regime-film-binary-features"

# Dataset selector. "fi2010" pulls the Ntakaris+2018 NoAuction DecPre CF files
# and trains on a public LOB benchmark. "polymarket" reproduces the original
# SQLite pipeline. Both share the same Mamba3 world-model code.
DATASET = "polymarket"
assert DATASET in ("polymarket", "fi2010"), DATASET

WORK_ROOT = Path('/content') if IS_COLAB else Path.home() / 'findrama_workspace'
PROJECT_DIR = str(WORK_ROOT / 'Drama') if IS_COLAB else str(Path.cwd())
CACHE_ROOT = WORK_ROOT / 'findrama_cache'
DATA_ROOT = WORK_ROOT / 'findrama_data'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Data lives in the dataset repo (data/ + logs/ only). Checkpoints and prebuilt
# CUDA wheels now live in dedicated repos so the dataset repo stays data-only.
HF_REPO = "sj-hryi/FinDrama"
CKPT_REPO = "sj-hryi/FinDrama-checkpoints"
WHEELS_REPO = "sj-hryi/FinDrama-wheels"
HF_REVISION = None
FORCE_REBUILD_WHEELS = False

# Default "Run All" path: full Mamba3 MIMO pretraining on the active dataset.
# Flip RUN_PROBES to True if you want the 3-probe collapse-diagnosis sweep
# instead. SMOKE_TEST clips both probes and pretrain to 20 steps for plumbing
# verification.
SMOKE_TEST = False
RUN_PROBES = False
HOURS_TRAIN = 6
HOURS_VAL = 1
PROBE_STEPS = 20 if SMOKE_TEST else 1000
MAX_STEPS = 20 if SMOKE_TEST else 8000

# Map dataset choice to the right config file. Both files live in src/config_files/.
CONFIG_FILENAME_BY_DATASET = {
    "polymarket": "configure_lob.yaml",
    "fi2010": "configure_fi2010.yaml",
}
CONFIG_FILENAME = CONFIG_FILENAME_BY_DATASET[DATASET]

def pip_install(*args):
    """Jupyter-safe pip wrapper. Avoids shell=True quirks on Windows."""
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def get_hf_token():
    """Resolve HF_TOKEN: Colab Secrets first when on Colab, env var otherwise."""
    if IS_COLAB:
        try:
            from google.colab import userdata
            t = userdata.get('HF_TOKEN')
            if t:
                return t
        except Exception:
            pass
    return os.environ.get('HF_TOKEN')

print(f"Runtime: {'Colab' if IS_COLAB else 'local'}, project at {PROJECT_DIR}")
print(f"Dataset: {DATASET}, config: {CONFIG_FILENAME}")
print(f"Mode: {'SMOKE_TEST' if SMOKE_TEST else 'probes' if RUN_PROBES else 'full pretrain'} | MAX_STEPS={MAX_STEPS}")


In [ ]:
HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing. On Colab add it to Secrets (key icon, left sidebar). "
        "On local set the HF_TOKEN env var before launching jupyter."
    )

pip_install('huggingface_hub')
from huggingface_hub import snapshot_download

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATASET == "polymarket":
    # Bundles live under data/polymarket/{train,validation,test}.tar.gz on the HF
    # mirror. Each .tar.gz holds polymarket.db + polymarket_books/ + binance_lob/.
    # Training uses train + validation; the test split is kept on HF for separate
    # evaluation and is not pulled here.
    print('Pulling Polymarket dataset bundle from HuggingFace Hub.')
    snapshot_download(
        repo_id=HF_REPO, repo_type='dataset',
        allow_patterns=['data/polymarket/train.tar.gz', 'data/polymarket/validation.tar.gz'],
        revision=HF_REVISION, token=HF_TOKEN,
        local_dir=str(DATA_ROOT),
    )
    TRAIN_GZ = DATA_ROOT / 'data' / 'polymarket' / 'train.tar.gz'
    VAL_GZ = DATA_ROOT / 'data' / 'polymarket' / 'validation.tar.gz'
    for p in (TRAIN_GZ, VAL_GZ):
        if not p.exists():
            raise FileNotFoundError(p)
    print('Polymarket bundles ready:', TRAIN_GZ.name, VAL_GZ.name)
elif DATASET == "fi2010":
    # FI-2010 NoAuction DecPre CF files (Ntakaris+2018), 3-way split on the HF
    # mirror: train/ (85% of the original train file), validation/ (carved last
    # 15%), test/ (the original Test_ file). Training uses train + validation;
    # the test split lives at data/fi2010/test/ on HF and is not pulled here.
    print('Pulling FI-2010 NoAuction DecPre CF files from HuggingFace Hub.')
    snapshot_download(
        repo_id=HF_REPO, repo_type='dataset',
        allow_patterns=[
            'data/fi2010/train/Train_Dst_NoAuction_DecPre_CF_7.txt',
            'data/fi2010/validation/Val_Dst_NoAuction_DecPre_CF_7.txt',
        ],
        revision=HF_REVISION, token=HF_TOKEN,
        local_dir=str(DATA_ROOT),
    )
    FI2010_TRAIN = DATA_ROOT / 'data' / 'fi2010' / 'train' / 'Train_Dst_NoAuction_DecPre_CF_7.txt'
    FI2010_VAL = DATA_ROOT / 'data' / 'fi2010' / 'validation' / 'Val_Dst_NoAuction_DecPre_CF_7.txt'
    for p in (FI2010_TRAIN, FI2010_VAL):
        if not p.exists():
            raise FileNotFoundError(
                f"FI-2010 file missing at {p}. The mirror lives at "
                "sj-hryi/FinDrama under data/fi2010/{train,validation,test}/."
            )
    print('FI-2010 files ready:', FI2010_TRAIN.name, FI2010_VAL.name)
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")


In [ ]:
# Colab: clone fresh into /content/Drama. Local: assume the notebook is being
# run from inside an already-cloned repo, so just chdir.
if IS_COLAB:
    import shutil
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])
os.chdir(PROJECT_DIR)
print('Repo ready:', os.getcwd())


In [ ]:
os.chdir(PROJECT_DIR)

# Local runs assume the user has torch + causal_conv1d + mamba_ssm in their conda
# env. Colab runs install everything from scratch and use the HF wheel cache for
# the C++/CUDA extensions.
if IS_COLAB:
    pip_cache = CACHE_ROOT / 'pip'
    pip_cache.mkdir(parents=True, exist_ok=True)
    os.environ['PIP_CACHE_DIR'] = str(pip_cache)

    pip_install('--upgrade', 'pip')
    pip_install('huggingface_hub[cli]')
    pip_install('packaging', 'ninja', 'setuptools==69.5.1', 'numpy>=2,<3')
    pip_install('torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
                '--index-url', 'https://download.pytorch.org/whl/cu124')

    import torch
    cuda_version = (torch.version.cuda or 'cpu').replace('.', '')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
    else:
        major, minor = 0, 0
    arch_list = f"{major}.{minor}"
    gpu_name = f"sm{major}{minor}"
    os.environ['TORCH_CUDA_ARCH_LIST'] = arch_list

    wheelhouse = CACHE_ROOT / f"wheels-py{sys.version_info.major}{sys.version_info.minor}-torch260-cu{cuda_version}-{gpu_name}"
    wheelhouse.mkdir(parents=True, exist_ok=True)
    print('Wheel cache:', wheelhouse, 'TORCH_CUDA_ARCH_LIST:', arch_list)

    from huggingface_hub import snapshot_download, HfApi
    # Prebuilt wheels live in the dedicated WHEELS_REPO (dataset), one folder per
    # python/torch/cuda/arch combo, matching wheelhouse.name.
    print(f"Pulling cached wheels from HF Hub ({wheelhouse.name}) ...")
    snapshot_download(
        repo_id=WHEELS_REPO, repo_type='dataset',
        allow_patterns=f"{wheelhouse.name}/*.whl",
        local_dir=str(CACHE_ROOT), token=HF_TOKEN,
    )

    def latest_wheel(prefix):
        wheels = sorted(wheelhouse.glob(f"{prefix}-*.whl"), key=lambda p: p.stat().st_mtime, reverse=True)
        return wheels[0] if wheels else None

    def upload_wheel(wheel):
        if not HF_TOKEN:
            return
        HfApi().upload_file(
            path_or_fileobj=str(wheel),
            path_in_repo=f"{wheelhouse.name}/{wheel.name}",
            repo_id=WHEELS_REPO, repo_type='dataset', token=HF_TOKEN,
        )
        print('Uploaded wheel to HF Hub:', wheel.name)

    def build_or_install(prefix, build_cmd, package_name):
        wheel = latest_wheel(prefix)
        if FORCE_REBUILD_WHEELS or wheel is None:
            print(f"Building {package_name} wheel.")
            subprocess.check_call(build_cmd, shell=True, env=os.environ)
            wheel = latest_wheel(prefix)
            if wheel is None:
                raise FileNotFoundError(f"No {prefix} wheel found in {wheelhouse}")
            upload_wheel(wheel)
        else:
            print(f"Using cached {package_name} wheel: {wheel.name}")
        pip_install('--force-reinstall', '--no-deps', str(wheel))

    build_or_install('causal_conv1d',
                     f"pip wheel -q --no-deps --no-build-isolation --wheel-dir {wheelhouse} 'causal-conv1d>=1.4.0'",
                     'causal-conv1d')
    build_or_install('mamba_ssm',
                     f"MAMBA_FORCE_BUILD=TRUE pip wheel -q --no-deps --no-build-isolation --wheel-dir {wheelhouse} git+https://github.com/state-spaces/mamba.git",
                     'mamba-ssm')
    pip_install('-r', 'requirements.txt')
    pip_install('tilelang')
    pip_install('--upgrade', 'triton')

import torch, numpy as np
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.version.cuda)
print('numpy', np.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Some torch builds ship pytorch-triton without set_allocator; mamba_ssm's
# import path requires the symbol to exist.
import triton
if not hasattr(triton, 'set_allocator'):
    triton.set_allocator = lambda fn: None

import causal_conv1d_cuda
from mamba_ssm.modules.mamba3 import Mamba3
print('Mamba3 imported:', Mamba3)
try:
    from mamba_ssm.ops.tilelang.mamba3.mamba3_mimo import mamba3_mimo
    print('Mamba3 MIMO TileLang import: ok', mamba3_mimo)
except Exception as exc:
    print('Mamba3 MIMO TileLang import: unavailable', repr(exc))


In [ ]:
import shutil, tarfile

project = Path(PROJECT_DIR)
data_dir = project / 'data'
train_dir = data_dir / 'train'
val_dir = data_dir / 'validation'

def _is_metadata_path(path):
    parts = [str(x) for x in Path(path).parts]
    return any(part == '__MACOSX' or part.startswith('._') for part in parts)

def _extract_targz(archive, out):
    out.mkdir(parents=True, exist_ok=True)
    print('Extracting', archive, '->', out)
    with tarfile.open(archive, 'r:gz') as tf:
        members = [m for m in tf.getmembers() if not _is_metadata_path(m.name)]
        try:
            tf.extractall(out, members=members, filter='data')
        except TypeError:
            tf.extractall(out, members=members)

def extract_polymarket_bundle():
    # Each .tar.gz holds polymarket.db + polymarket_books/ + binance_lob/ at its
    # root. The loader reads <split>/polymarket.db and <split>/polymarket_books/,
    # so extract each bundle straight into data/<split>/.
    if (train_dir / 'polymarket.db').exists() and (val_dir / 'polymarket.db').exists():
        print('Polymarket data already prepared at', data_dir)
        return
    for split, gz in (('train', TRAIN_GZ), ('validation', VAL_GZ)):
        out = data_dir / split
        shutil.rmtree(out, ignore_errors=True)
        _extract_targz(gz, out)
    print('Prepared train:', train_dir)
    print('Prepared validation:', val_dir)

def stage_fi2010_files():
    # The loader expects data/<split>/<filename>.txt under the repo. Copy the
    # downloaded files there so we can pass --data-train data/train --data-val
    # data/validation just like the Polymarket path.
    src_train = DATA_ROOT / 'data' / 'fi2010' / 'train' / 'Train_Dst_NoAuction_DecPre_CF_7.txt'
    src_val = DATA_ROOT / 'data' / 'fi2010' / 'validation' / 'Val_Dst_NoAuction_DecPre_CF_7.txt'
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    dst_train = train_dir / src_train.name
    dst_val = val_dir / src_val.name
    if not dst_train.exists():
        shutil.copy(src_train, dst_train)
    if not dst_val.exists():
        shutil.copy(src_val, dst_val)
    print('Prepared FI-2010 train:', dst_train)
    print('Prepared FI-2010 validation:', dst_val)

if DATASET == "polymarket":
    extract_polymarket_bundle()
elif DATASET == "fi2010":
    stage_fi2010_files()
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")


In [ ]:
import re, datetime
from pathlib import Path

os.chdir(PROJECT_DIR)

# Each probe flips one hardcoded constant from default to test a collapse hypothesis.
# H1 raises the representation KL weight to match DreamerV3 0.5/0.5 symmetry.
# H2 zeros the free-bits floor to free the dynamics signal from the 1-nat clip.
# H3 lowers the decoder size weight so price features can compete with depth features.
PROBES = [
    ("H1_rep_loss_weight_0p5", ['--Models.WorldModel.RepresentationLossWeight', '0.5']),
    ("H2_free_bits_0p0",       ['--Models.WorldModel.FreeBits', '0.0']),
    ("H3_size_weight_1p0",     ['--Models.WorldModel.Decoder.SizeWeight', '1.0']),
]

RUN_DATE = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
LOG_DIR = WORK_ROOT / 'probe_logs' / RUN_DATE
LOG_DIR.mkdir(parents=True, exist_ok=True)
STDOUT_PATH = LOG_DIR / 'stdout.txt'

# Every run's stdout is appended to one shared log file. Tqdm progress bars
# are thinned to GPU-util cadence to keep the file small.
_log_handle = STDOUT_PATH.open('w', encoding='utf-8')
_TQDM_RE = re.compile(r'^pretrain:\s*\d+%\|.*\|\s*(\d+)/(\d+)\s*\[')
_GPU_RE = re.compile(r'\[GPU\] util=')
_state = {'gpu_seen_since_last_progress': True}

def _log_filtered(line):
    print(line, end='')
    if _GPU_RE.search(line):
        _state['gpu_seen_since_last_progress'] = True
    m = _TQDM_RE.match(line)
    if m:
        step, total = int(m.group(1)), int(m.group(2))
        is_endpoint = step <= 1 or step == total
        if is_endpoint or _state['gpu_seen_since_last_progress']:
            _log_handle.write(line)
            _state['gpu_seen_since_last_progress'] = False
        return
    _log_handle.write(line)
    _log_handle.flush()

CONFIG_PATH = Path(PROJECT_DIR) / 'src' / 'config_files' / CONFIG_FILENAME

# Mamba3 MIMO is enabled by default in both configs. The TileLang MIMO kernel
# is the fast path on H100 (sm_90); other GPUs fall back to the Python reference
# automatically with a log warning. No notebook-level override needed.
def run_train(steps, extra_args, label):
    _log_filtered(f"\n{'='*72}\n{label}\n{'='*72}\n")
    cmd = [
        sys.executable, '-u', '-B', 'src/train_lob.py',
        '--config', str(CONFIG_PATH),
        '--dataset', DATASET,
        '--hours-train', str(HOURS_TRAIN),
        '--hours-val', str(HOURS_VAL),
        *extra_args,
    ]
    _log_filtered('Running: ' + ' '.join(cmd) + '\n')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        _log_filtered(line)
    rc = proc.wait()
    if rc:
        raise subprocess.CalledProcessError(rc, cmd)

run_names = []
try:
    if RUN_PROBES:
        for name, extra_args in PROBES:
            run_names.append(name)
            run_train(PROBE_STEPS, extra_args, f"Probe {name} ({PROBE_STEPS} steps)")
    else:
        run_names.append('full_pretrain')
        # run_train(MAX_STEPS, [], f"Full Mamba3 MIMO pretrain ({MAX_STEPS} steps)")
        run_train(MAX_STEPS, ['--Models.WorldModel.Mamba3.is_mimo', 'false'], f"Full Mamba3 SISO pretrain ({MAX_STEPS} steps)")
finally:
    _log_handle.close()

# Recover wandb run ids from the log in chronological order. De-dup while
# preserving order; each wandb.init() prints the line a few times.
log_text = STDOUT_PATH.read_text(encoding='utf-8')
ordered_run_ids = []
for r in re.findall(r'offline-run-\d+_\d+-(\w+)', log_text):
    if r not in ordered_run_ids:
        ordered_run_ids.append(r)

# Copy each run's wandb-summary.json next to the stdout log.
WANDB_ROOT = Path(PROJECT_DIR) / 'wandb'
for name, run_id in zip(run_names, ordered_run_ids):
    matches = list(WANDB_ROOT.glob(f'offline-run-*-{run_id}/files/wandb-summary.json'))
    if not matches:
        print(f"warning: no wandb-summary.json for {name} ({run_id})")
        continue
    dst = LOG_DIR / f"{name}_{run_id}_summary.json"
    shutil.copy(matches[0], dst)
    print(f"copied {matches[0].name} -> {dst.name}")

# Upload the cell-run log dir to HF Hub under logs/, alongside checkpoints/.
hf_token = get_hf_token()
if hf_token and HF_REPO:
    from huggingface_hub import HfApi
    HfApi().upload_folder(
        folder_path=str(LOG_DIR),
        path_in_repo=f"logs/{RUN_DATE}",
        repo_id=HF_REPO, repo_type='dataset', token=hf_token,
    )
    print(f"\nUploaded run logs to https://huggingface.co/datasets/{HF_REPO}/tree/main/logs/{RUN_DATE}")
else:
    print(f"\nHF_TOKEN missing; logs only saved locally at {LOG_DIR}")


In [ ]:
# Read-only dump of every offline wandb run's full summary. Use this to recover
# Imagine/mid_norm_std and per-feature MSE values that wandb truncates with
# `+1 ...` in its CLI summary block.
import json
wandb_root = Path(PROJECT_DIR) / 'wandb'
for d in sorted(wandb_root.glob('offline-run-*')):
    p = d / 'files' / 'wandb-summary.json'
    print('==', d, '==')
    if p.exists():
        with p.open() as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print(f"(no summary file at {p})")
    print()


In [ ]:
src = Path(PROJECT_DIR) / 'saved_models' / 'lob'
if not src.exists():
    print('No checkpoint directory found yet:', src)
else:
    hf_token = get_hf_token()
    if hf_token and CKPT_REPO:
        from huggingface_hub import HfApi
        HfApi().upload_folder(
            folder_path=str(src),
            path_in_repo='checkpoints/lob',
            repo_id=CKPT_REPO, repo_type='model', token=hf_token,
        )
        print('Backed up checkpoints to HF Hub:', CKPT_REPO)
    else:
        print('HF_TOKEN not set - checkpoints only saved locally at:', src)
